# OOP Part 4 — Abstraction
### 20 Questions

**The core idea:** Abstraction means defining WHAT a class must do without necessarily saying HOW — and, importantly here, making Python actually ENFORCE that subclasses implement the required pieces, instead of just hoping they remember to (which is all `NotImplementedError` alone gives you, as you saw at the end of the last notebook).

Python's tool for this is the `abc` module (Abstract Base Classes): `ABC` and `@abstractmethod`.

Same rules: attempt first, run your own cell, then compare.

---

## Part A: The Problem Abstraction Solves (Q1-Q3)

**Q1** Recall the `RiskModel` pattern from the last notebook — a base class where `score()` raises `NotImplementedError`. Recreate it, then create a "bad" subclass `BrokenModel(RiskModel)` that FORGETS to override `score()` at all (just `pass`). Create an instance of `BrokenModel` — notice Python lets you create it with ZERO complaints, and the problem only shows up LATER when you actually call `.score()`.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
class RiskModel:
    def score(self, applicant):
        raise NotImplementedError("Subclasses must implement score()")

class BrokenModel(RiskModel):
    pass

m = BrokenModel()  # No error here — this is the problem
print("Instance created successfully:", m)

try:
    m.score({"credit_score": 700})
except NotImplementedError as e:
    print("Error only appears now, when called:", e)

**Q2** Explain in a comment why this delayed failure is risky in a real project — think about WHERE in a program's lifecycle the bug would actually surface versus where the mistake was actually made.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
# The mistake (forgetting to implement score()) happens at CLASS DEFINITION
# time, but the failure only surfaces at CALL time — which could be much
# later, in a different part of the codebase, possibly in production,
# long after the class was written. If BrokenModel sits unused for months
# before someone finally calls .score() on it, the bug is discovered far
# from its source, making it harder to trace and more likely to cause
# real damage (e.g. a batch job crashing mid-run instead of failing fast
# at startup).
print("See comment above")

**Q3** Import `ABC` and `abstractmethod` from the `abc` module: `from abc import ABC, abstractmethod`. Just import them for now — next questions will use them.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
from abc import ABC, abstractmethod
print("Imported")

## Part B: Building a Real Abstract Base Class (Q4-Q10)

**Q4** Redefine `RiskModel` to INHERIT from `ABC` (instead of nothing), and decorate `score(self, applicant)` with `@abstractmethod` (body can just be `pass`). Try creating a PLAIN `RiskModel()` instance directly — confirm this now raises `TypeError` IMMEDIATELY, before you even get to calling any method.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

try:
    m = RiskModel()
except TypeError as e:
    print("Error:", e)

**Q5** Now try the SAME "forgetting to implement" mistake from Q1: define `BrokenModel(RiskModel)` with just `pass` (no `score` override). Try to create an instance — confirm Python now catches this IMMEDIATELY at instantiation time, not later when called.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

class BrokenModel(RiskModel):
    pass

try:
    m = BrokenModel()
except TypeError as e:
    print("Error:", e)

**Q6** Define `SimpleScoreModel(RiskModel)` that PROPERLY implements `score(self, applicant)` (returning `applicant["credit_score"]`). Confirm you CAN now create an instance successfully, since the required method is implemented.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

class SimpleScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"]

m = SimpleScoreModel()
print(m.score({"credit_score": 720}))

**Q7** Add a SECOND abstract method `model_name(self)` to `RiskModel` (also decorated `@abstractmethod`). Confirm that `SimpleScoreModel` from Q6 (which only implements `score`, not `model_name`) now FAILS to instantiate — ALL abstract methods must be implemented, not just some.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

    @abstractmethod
    def model_name(self):
        pass

class SimpleScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"]

try:
    m = SimpleScoreModel()
except TypeError as e:
    print("Error:", e)

**Q8** Fix `SimpleScoreModel` by ALSO implementing `model_name(self)` (returning `"Simple Score Model"`). Confirm instantiation now succeeds and both methods work.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

    @abstractmethod
    def model_name(self):
        pass

class SimpleScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"]

    def model_name(self):
        return "Simple Score Model"

m = SimpleScoreModel()
print(m.model_name(), m.score({"credit_score": 720}))

**Q9** An abstract base class CAN also have regular (non-abstract) methods with real implementations that subclasses inherit for free. Add a CONCRETE method `describe(self, applicant)` to `RiskModel` that returns `f"{self.model_name()}: {self.score(applicant)}"` — notice it calls the ABSTRACT methods internally, trusting that whatever subclass exists will have implemented them. Test it on `SimpleScoreModel`.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

    @abstractmethod
    def model_name(self):
        pass

    def describe(self, applicant):
        return f"{self.model_name()}: {self.score(applicant)}"

class SimpleScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"]

    def model_name(self):
        return "Simple Score Model"

m = SimpleScoreModel()
print(m.describe({"credit_score": 720}))

**Q10** Explain in a comment WHY it's useful for an abstract class to have BOTH abstract methods (must be implemented per subclass) AND concrete methods (shared logic, written once) — what does `describe()` from Q9 demonstrate about this combination?

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
# Abstract methods define the CONTRACT — what every subclass MUST provide,
# since each risk model calculates score/name differently.
#
# Concrete methods let you write SHARED logic ONCE in the base class,
# reused by every subclass automatically. describe() doesn't need to know
# which specific model it's running on — it just calls self.model_name()
# and self.score(), trusting the contract is fulfilled (which ABC now
# guarantees at instantiation time). This avoids every subclass having to
# reimplement its own version of describe() with identical logic.
print("See comment above")

## Part C: Multiple Concrete Subclasses (Q11-Q15)

**Q11** Define `WeightedScoreModel(RiskModel)` implementing both required abstract methods: `score` returns `applicant["credit_score"] * 0.7 + applicant["income"] / 1000 * 0.3`, `model_name` returns `"Weighted Score Model"`. Test `.describe()` on it with a sample applicant.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

    @abstractmethod
    def model_name(self):
        pass

    def describe(self, applicant):
        return f"{self.model_name()}: {self.score(applicant)}"

class WeightedScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"] * 0.7 + applicant["income"] / 1000 * 0.3

    def model_name(self):
        return "Weighted Score Model"

m = WeightedScoreModel()
print(m.describe({"credit_score": 720, "income": 45000}))

**Q12** Create a LIST containing instances of BOTH `SimpleScoreModel` and `WeightedScoreModel` (recreate both classes). Loop through and call `.describe()` on each with the SAME applicant, printing results — this is polymorphism working ON TOP of the abstraction guarantee: you KNOW every item in the list has these methods.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass

    @abstractmethod
    def model_name(self):
        pass

    def describe(self, applicant):
        return f"{self.model_name()}: {self.score(applicant)}"

class SimpleScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"]
    def model_name(self):
        return "Simple Score Model"

class WeightedScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"] * 0.7 + applicant["income"] / 1000 * 0.3
    def model_name(self):
        return "Weighted Score Model"

models = [SimpleScoreModel(), WeightedScoreModel()]
applicant = {"credit_score": 720, "income": 45000}

for m in models:
    print(m.describe(applicant))

**Q13** Write a standalone function `pick_best_model(models, applicant)` that takes a LIST of `RiskModel` instances and returns the one with the HIGHEST `.score(applicant)`, using `max()` with a `key=` lambda. Test it on the Q12 list.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass
    @abstractmethod
    def model_name(self):
        pass

class SimpleScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"]
    def model_name(self):
        return "Simple Score Model"

class WeightedScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"] * 0.7 + applicant["income"] / 1000 * 0.3
    def model_name(self):
        return "Weighted Score Model"

def pick_best_model(models, applicant):
    return max(models, key=lambda m: m.score(applicant))

models = [SimpleScoreModel(), WeightedScoreModel()]
applicant = {"credit_score": 720, "income": 45000}
best = pick_best_model(models, applicant)
print(best.model_name())

**Q14** Add a THIRD abstract method `required_fields(self)` to `RiskModel` that should return a LIST of applicant dict keys the model needs. Implement it in both `SimpleScoreModel` (`["credit_score"]`) and `WeightedScoreModel` (`["credit_score", "income"]`). Add a CONCRETE method `validate(self, applicant)` to `RiskModel` that checks all `required_fields()` exist as keys in `applicant`, returning True/False. Test validate on an incomplete applicant dict.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass
    @abstractmethod
    def model_name(self):
        pass
    @abstractmethod
    def required_fields(self):
        pass

    def validate(self, applicant):
        return all(field in applicant for field in self.required_fields())

class SimpleScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"]
    def model_name(self):
        return "Simple Score Model"
    def required_fields(self):
        return ["credit_score"]

class WeightedScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"] * 0.7 + applicant["income"] / 1000 * 0.3
    def model_name(self):
        return "Weighted Score Model"
    def required_fields(self):
        return ["credit_score", "income"]

m = WeightedScoreModel()
print(m.validate({"credit_score": 720}))
print(m.validate({"credit_score": 720, "income": 45000}))

**Q15** Combine `validate()` and `score()`: write a concrete method `safe_score(self, applicant)` on `RiskModel` that returns `self.score(applicant)` if `self.validate(applicant)` is True, otherwise returns `None`. Test it with a valid and invalid applicant on `WeightedScoreModel`.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
from abc import ABC, abstractmethod

class RiskModel(ABC):
    @abstractmethod
    def score(self, applicant):
        pass
    @abstractmethod
    def required_fields(self):
        pass

    def validate(self, applicant):
        return all(field in applicant for field in self.required_fields())

    def safe_score(self, applicant):
        if self.validate(applicant):
            return self.score(applicant)
        return None

class WeightedScoreModel(RiskModel):
    def score(self, applicant):
        return applicant["credit_score"] * 0.7 + applicant["income"] / 1000 * 0.3
    def required_fields(self):
        return ["credit_score", "income"]

m = WeightedScoreModel()
print(m.safe_score({"credit_score": 720, "income": 45000}))
print(m.safe_score({"credit_score": 720}))

## Part D: Abstraction vs Duck Typing — Choosing the Right Tool (Q16-Q17)

**Q16** Explain in a comment: given that Python ALSO supports duck typing (any object with the right methods works, no formal class relationship needed, as you saw in the polymorphism notebook), when would you reach for a formal `ABC` instead of just relying on duck typing?

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
# Duck typing is fine for quick, informal cases, or when you don't control
# all the classes involved (e.g. working with third-party objects that
# happen to have compatible methods).
#
# Reach for ABC when: (1) you're designing a system where OTHER developers
# (or future you) will build new subclasses, and you want a clear, ENFORCED
# contract so mistakes are caught immediately rather than at some unpredictable
# later call; (2) you want the class hierarchy itself to document the
# required interface (someone reading RiskModel immediately sees exactly
# what score(), model_name(), required_fields() must do); (3) you have shared
# concrete logic (like validate()/safe_score()) that depends on those abstract
# methods being implemented correctly.
print("See comment above")

**Q17** Try to instantiate `ABC` directly: `from abc import ABC; a = ABC()`. Confirm this actually succeeds (unlike a class with `@abstractmethod`s) — `ABC` by itself is just a marker base class; it's the PRESENCE of `@abstractmethod`s that blocks instantiation, not inheriting from `ABC` alone.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
from abc import ABC
a = ABC()
print(a)
# ABC alone has no abstract methods, so there's nothing blocking instantiation.
# The enforcement comes specifically from having at least one @abstractmethod
# that hasn't been overridden.

## Part E: Mini Challenge (Q18-Q20)

**Q18** Design an abstract base class `NotificationChannel(ABC)` with abstract methods `send(self, message)` and `channel_name(self)`, plus a concrete method `send_with_log(self, message)` that prints `f"[{self.channel_name()}] Sending: {message}"` THEN calls `self.send(message)`. Implement TWO subclasses: `EmailChannel` (send just prints `f"Email sent: {message}"`) and `SMSChannel` (send prints `f"SMS sent: {message}"`). Test `.send_with_log()` on both.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
from abc import ABC, abstractmethod

class NotificationChannel(ABC):
    @abstractmethod
    def send(self, message):
        pass
    @abstractmethod
    def channel_name(self):
        pass

    def send_with_log(self, message):
        print(f"[{self.channel_name()}] Sending: {message}")
        self.send(message)

class EmailChannel(NotificationChannel):
    def send(self, message):
        print(f"Email sent: {message}")
    def channel_name(self):
        return "Email"

class SMSChannel(NotificationChannel):
    def send(self, message):
        print(f"SMS sent: {message}")
    def channel_name(self):
        return "SMS"

for channel in [EmailChannel(), SMSChannel()]:
    channel.send_with_log("Your payment is due")

**Q19** Add a THIRD channel `PushChannel(NotificationChannel)` — but deliberately FORGET to implement `channel_name()` (only implement `send`). Confirm attempting to create it raises `TypeError`, and print which specific method is missing based on the error message.

In [ ]:
# YOUR CODE HERE


**Solution 19**

In [ ]:
from abc import ABC, abstractmethod

class NotificationChannel(ABC):
    @abstractmethod
    def send(self, message):
        pass
    @abstractmethod
    def channel_name(self):
        pass

class PushChannel(NotificationChannel):
    def send(self, message):
        print(f"Push sent: {message}")

try:
    p = PushChannel()
except TypeError as e:
    print("Error:", e)

**Q20** The big one: build `alert_all_channels(channels, message)` that loops through a LIST of `NotificationChannel` instances and calls `.send_with_log()` on each, wrapping each call in try/except to catch ANY exception from a misbehaving channel (so one broken channel doesn't stop the others), printing `f"Failed on {type(c).__name__}"` on failure. Create a properly-fixed `PushChannel` plus `EmailChannel`/`SMSChannel`, put all three in a list, and run the function.

In [ ]:
# YOUR CODE HERE


**Solution 20**

In [ ]:
from abc import ABC, abstractmethod

class NotificationChannel(ABC):
    @abstractmethod
    def send(self, message):
        pass
    @abstractmethod
    def channel_name(self):
        pass

    def send_with_log(self, message):
        print(f"[{self.channel_name()}] Sending: {message}")
        self.send(message)

class EmailChannel(NotificationChannel):
    def send(self, message):
        print(f"Email sent: {message}")
    def channel_name(self):
        return "Email"

class SMSChannel(NotificationChannel):
    def send(self, message):
        print(f"SMS sent: {message}")
    def channel_name(self):
        return "SMS"

class PushChannel(NotificationChannel):
    def send(self, message):
        print(f"Push sent: {message}")
    def channel_name(self):
        return "Push"

def alert_all_channels(channels, message):
    for c in channels:
        try:
            c.send_with_log(message)
        except Exception:
            print(f"Failed on {type(c).__name__}")

channels = [EmailChannel(), SMSChannel(), PushChannel()]
alert_all_channels(channels, "Your account balance is low")

---
## Checkpoint

20 questions: the exact problem abstraction solves (delayed vs immediate failure), building a real `ABC` with `@abstractmethod`, mixing abstract methods (the contract) with concrete methods (shared logic built on top of that contract), multiple properly-implemented subclasses working together polymorphically, when to choose formal `ABC` over duck typing, and a mini challenge building a notification-channel system.

**You've now completed the full OOP arc:** classes/objects → encapsulation & static/classmethods → inheritance & polymorphism → abstraction. Together these four notebooks cover essentially everything you'll encounter reading library source code (including pandas/sklearn internals, if you ever look) and everything you'd need to structure your own credit-risk project as clean, extensible classes instead of one long script.

Ready for the OOP project now, or want to review anything first?